# 隨機市場模擬研究

> **核心命題**：純隨機的價格路徑，天然會形成支撐阻力與圖形型態。
> 這不是市場智慧，而是隨機過程的數學副產品。

In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import find_peaks

from market_simulator import MarketSimulator, SimConfig
from pattern_visualizer import PatternVisualizerAuto
from utils import plot_volume_profile, compute_occupation_time, detect_sr_levels, random_walk_baseline

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (16, 6)
print('Environment ready.')

## Step 2: 上帝擲骰子 — 生成隨機市場數據

In [ ]:
# 取消 seed 讓每次執行都產生不同數據
# np.random.seed(42)

sim = MarketSimulator(start_price=100, mode='orderbook')
GLOBAL_PRICE_DATA = sim.run(steps=100_000)

print(f'Generated {len(GLOBAL_PRICE_DATA):,} ticks')
print(f'Price range: {GLOBAL_PRICE_DATA.min():.2f} — {GLOBAL_PRICE_DATA.max():.2f}')
print(f'Data locked in GLOBAL_PRICE_DATA.')

## Step 3: Volume Profile + 支撐阻力偵測

隨機路徑在某些價格帶停留時間更長 → 直方圖高峰 → 被偵測為支撐阻力

In [ ]:
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18, 7))
gs = gridspec.GridSpec(1, 4, figure=fig)
ax1 = fig.add_subplot(gs[0, :3])
ax2 = fig.add_subplot(gs[0, 3])

plot_volume_profile(GLOBAL_PRICE_DATA, ax1, ax2, bins=50, highlight_sr=True)
fig.suptitle('Phase 1: Random Walk + Liquidity Walls', fontsize=14, color='white')
plt.tight_layout()
plt.show()

## Step 4: 自動型態偵測

純隨機數據上，自動偵測到頭肩頂、雙底、趨勢線

In [ ]:
viz = PatternVisualizerAuto(GLOBAL_PRICE_DATA, order=40)

fig, ax = plt.subplots(figsize=(18, 7))
viz.plot_base_chart(ax, color='white', label='Random Data', subsample=5)
detected = viz.find_and_draw_patterns(ax, max_patterns=8)

ax.set_title('Phase 2: Patterns Auto-Detected on Random Data', fontsize=14, color='white')
ax.grid(True, alpha=0.1)
ax.legend(fontsize=8)

print(f'Detected {len(detected)} patterns on pure random data.')
for p in detected:
    print(f"  - {p['type']}")
plt.tight_layout()
plt.show()

## Step 5: Occupation Time — 支撐阻力的統計根源

隨機遊走的路徑在某些價格區間停留的時間不均勻分佈，
停留時間越長的區間，就會被識別為支撐或阻力。

In [ ]:
df_occ = compute_occupation_time(GLOBAL_PRICE_DATA, bins=100)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 停留時間條形圖
axes[0].barh(df_occ['price_level'], df_occ['occupation_time'],
             height=(df_occ['price_level'].max() - df_occ['price_level'].min()) / 100,
             color='orange', alpha=0.6)
axes[0].set_xlabel('Occupation Time (ticks)')
axes[0].set_ylabel('Price Level')
axes[0].set_title('Occupation Time Distribution')
axes[0].grid(alpha=0.1)

# 最高停留時間的前10個價位
top10 = df_occ.nlargest(10, 'occupation_time')
axes[1].barh(top10['price_level'].astype(str), top10['occupation_time'], color='red', alpha=0.7)
axes[1].set_title('Top 10 Longest-Stayed Price Levels (= S/R)')
axes[1].grid(alpha=0.1)

plt.tight_layout()
plt.show()

## Step 6: 三種模型對比

GBM / OrderBook / Pure Random Walk — 三者都能產出視覺上相似的走勢

In [ ]:
steps = 50_000
gbm_data = MarketSimulator(mode='gbm',       config=SimConfig(steps=steps)).run()
ob_data  = MarketSimulator(mode='orderbook', config=SimConfig(steps=steps)).run()
rw_data  = random_walk_baseline(steps)

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=False)
for ax, (data, title, color) in zip(axes, [
    (rw_data,  'Pure Random Walk',  'grey'),
    (gbm_data, 'GBM',               'cyan'),
    (ob_data,  'OrderBook Sim',      'orange'),
]):
    ax.plot(data, color=color, lw=0.6)
    for lvl in detect_sr_levels(data):
        ax.axhline(lvl, color='red', lw=0.5, alpha=0.4, linestyle='--')
    ax.set_title(title, color='white')
    ax.grid(alpha=0.1)

fig.suptitle('All Three Produce Visually Similar Patterns', fontsize=13, color='white')
plt.tight_layout()
plt.show()